# Dataset-level quality feature import workflow

该 notebook 用于在 Colab 中从 Google Drive 前序结果包生成数据集级质量 Inception 特征记录, 重建 FID / KID 治理产物, 并把 zip 保存回 Google Drive。当前入口按 `SLM_WM_PAPER_RUN_NAME` 选择 `pilot_paper` 或 `full_paper` 规模; 只有满足数据集级样本规模与正式特征导入边界时才支持 dataset-level FID / KID 论文声明。


In [ ]:
import os

from google.colab import drive

drive.mount('/content/drive')
# 修改为 "full_paper" 可切换完整论文运行层级。
SLM_WM_PAPER_RUN_NAME = "pilot_paper"
os.environ["SLM_WM_PAPER_RUN_NAME"] = SLM_WM_PAPER_RUN_NAME


In [ ]:
import importlib.metadata as importlib_metadata
import importlib.util


def package_version_or_missing(package_name):
    try:
        return importlib_metadata.version(package_name)
    except importlib_metadata.PackageNotFoundError:
        return 'not_installed'


dependency_report = {
    'torch': package_version_or_missing('torch'),
    'torchvision': package_version_or_missing('torchvision'),
    'scipy': package_version_or_missing('scipy'),
    'pillow_available': importlib.util.find_spec('PIL') is not None,
}
print(dependency_report)


In [ ]:
import os
import subprocess
import sys

repository_url = os.environ.get('SLM_WM_REPOSITORY_URL', 'https://github.com/RICHAAARC/SLM-WM.git')
repository_ref = os.environ.get('SLM_WM_REPOSITORY_REF', 'main')
workspace_dir = os.environ.get('SLM_WM_WORKSPACE_DIR', '/content/slm_wm_repository')

if not os.path.exists(workspace_dir):
    subprocess.run(['git', 'clone', repository_url, workspace_dir], check=True)

subprocess.run(['git', 'fetch', '--all'], cwd=workspace_dir, check=True)
subprocess.run(['git', 'checkout', repository_ref], cwd=workspace_dir, check=True)
os.chdir(workspace_dir)
if workspace_dir not in sys.path:
    sys.path.insert(0, workspace_dir)
print({'workspace_dir': workspace_dir, 'repository_ref': repository_ref})

from paper_workflow.colab_utils.paper_run_environment import configure_paper_run_environment
paper_run_environment = configure_paper_run_environment(
    workflow_name="dataset_level_quality",
)
print(paper_run_environment)


In [ ]:
import os

from paper_workflow.colab_utils.dataset_level_quality import run_default_dataset_level_quality_from_drive_plan

dataset_level_quality_summary = run_default_dataset_level_quality_from_drive_plan(
    root='.',
    real_attack_evaluation_drive_dir=os.environ['SLM_WM_REAL_ATTACK_EVALUATION_DRIVE_DIR'],
    aligned_rescoring_drive_dir=os.environ['SLM_WM_ALIGNED_RESCORING_DRIVE_DIR'],
    formal_min_sample_count=int(os.environ['SLM_WM_FORMAL_MIN_SAMPLE_COUNT']),
)
dataset_level_quality_summary


In [ ]:
from pathlib import Path

summary_path = Path('outputs/dataset_level_quality/dataset_level_quality_result.json')
input_manifest_path = Path('outputs/dataset_level_quality/dataset_level_quality_input_package_manifest.json')
formal_import_report_path = Path('outputs/dataset_level_quality/dataset_quality_formal_feature_import_report.json')
print(summary_path.read_text(encoding='utf-8') if summary_path.exists() else dataset_level_quality_summary)
for path in (input_manifest_path, formal_import_report_path):
    if path.exists():
        print(path.read_text(encoding='utf-8'))


In [ ]:
import os
import subprocess
from datetime import datetime, timezone

from paper_workflow.colab_utils.dataset_level_quality import package_dataset_level_quality_outputs


def resolve_short_commit():
    try:
        result = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True, capture_output=True, text=True)
    except Exception:
        return 'git_unknown'
    return result.stdout.strip() or 'git_unknown'


drive_output_dir = os.environ['SLM_WM_DATASET_QUALITY_DRIVE_DIR']
utc_suffix = datetime.now(timezone.utc).strftime('%Y%m%dt%H%M%sz')
short_commit = resolve_short_commit()
archive_name = f'dataset_level_quality_package_{utc_suffix}_{short_commit}.zip'
archive_record = package_dataset_level_quality_outputs(root='.', drive_output_dir=drive_output_dir, archive_name=archive_name)
archive_record.to_dict()


In [ ]:
from pathlib import Path

for result_path in sorted(Path('outputs/dataset_level_quality').glob('*.json')):
    if result_path.is_file():
        print(result_path)
        print(result_path.read_text(encoding='utf-8')[:4000])

for result_path in sorted(Path('outputs/dataset_level_quality').glob('*.csv')):
    if result_path.is_file():
        print(result_path)
        print(result_path.read_text(encoding='utf-8')[:4000])

assert dataset_level_quality_summary['run_decision'] == 'pass', dataset_level_quality_summary
assert dataset_level_quality_summary['dataset_level_quality_proxy_ready'] is True, dataset_level_quality_summary
assert dataset_level_quality_summary['formal_feature_backend_ready'] is True, dataset_level_quality_summary
assert dataset_level_quality_summary['formal_sample_scale_ready'] is True, dataset_level_quality_summary
assert dataset_level_quality_summary['formal_fid_kid_ready'] is True, dataset_level_quality_summary
assert dataset_level_quality_summary['supports_paper_claim'] is False, dataset_level_quality_summary
assert archive_record.archive_digest == archive_record.drive_archive_digest, archive_record.to_dict()
